# LexRAG — RAGAS Evaluation Notebook

**Purpose:** Evaluate the LexRAG pipeline across 50 curated Indian legal questions using RAGAS metrics.  
**Judge LLM:** Llama 3.1 8B via Ollama (local, no rate limits, no API cost)  
**Production LLM:** Groq (used only in the pipeline for answer generation, not for evaluation)  

### Metrics evaluated
| Metric | What it measures |
|---|---|
| `Faithfulness` | Are all claims in the answer supported by the retrieved context? |
| `LLMContextRecall` | Did the retriever fetch the chunks needed to answer correctly? |
| `FactualCorrectness` | Does the answer match the ground truth factually? |
| `ResponseRelevancy` | Is the answer actually relevant to the question asked? |

### Prerequisites
```bash
# 1. Install Ollama
curl -fsSL https://ollama.ai/install.sh | sh

# 2. Pull the judge model (one-time, ~5GB)
ollama pull llama3.1:8b

# 3. Install Python dependencies
pip install ragas==0.2.15 langchain-ollama langchain-groq datasets pandas tqdm

# 4. Make sure your LexRAG backend is running
# cd backend && uvicorn main:app --port 8000
```

**Select kernel:** `LexRAG (Python 3.11)` before running.

---
## Cell 1 — Imports

In [1]:
import os
import sys
import time
import json
import requests
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from datasets import Dataset
from dotenv import load_dotenv

# RAGAS 0.2.x imports — correct class names for this version
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    LLMContextRecall,
    FactualCorrectness,
    ResponseRelevancy,
)

# Ollama via LangChain
from langchain_ollama import ChatOllama, OllamaEmbeddings

load_dotenv(dotenv_path=Path("..") / ".env")
print("✅ Imports successful")

✅ Imports successful


---
## Cell 2 — Verify Ollama is running

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "llama3.1:8b"   # change to llama3.1:70b if you have 32GB+ RAM
BACKEND_URL     = "http://localhost:8000"  # your FastAPI backend

# --- Check Ollama health ---
try:
    resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    models = [m["name"] for m in resp.json().get("models", [])]
    print(f"✅ Ollama is running. Available models: {models}")
    if not any(OLLAMA_MODEL.split(":")[0] in m for m in models):
        print(f"\n⚠️  Model '{OLLAMA_MODEL}' not found.")
        print(f"   Run this in a terminal: ollama pull {OLLAMA_MODEL}")
except Exception as e:
    print(f"❌ Ollama not reachable: {e}")
    print("   Start Ollama with: ollama serve")

# --- Check LexRAG backend health ---
try:
    resp = requests.get(f"{BACKEND_URL}/health", timeout=5)
    print(f"✅ LexRAG backend is running: {resp.json()}")
except Exception as e:
    print(f"❌ Backend not reachable: {e}")
    print("   Start it with: cd backend && uvicorn main:app --port 8000")

✅ Ollama is running. Available models: ['llama3.1:8b', 'llama3:latest', 'mistral:latest']
✅ LexRAG backend is running: {'status': 'ok', 'service': 'lexrag-api'}


---
## Cell 3 — Initialise RAGAS judge (Ollama, zero rate limits)

In [3]:
# LLM judge — Llama 3.1 8B running locally via Ollama
ollama_llm = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,          # deterministic evaluation
    num_predict=1024,       # max tokens per eval response
)

# Embeddings — used by ResponseRelevancy metric
ollama_embeddings = OllamaEmbeddings(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
)

# Wrap for RAGAS 0.2.x
evaluator_llm    = LangchainLLMWrapper(ollama_llm)
evaluator_embed  = LangchainEmbeddingsWrapper(ollama_embeddings)

# RunConfig — generous timeouts since local LLM is slower than API
run_config = RunConfig(
    max_workers=1,    # 1 = sequential, safe for local Ollama
    timeout=120,      # 2 minutes per eval call
    max_retries=3,
)

# Quick sanity check
test = ollama_llm.invoke("Reply with only the word: OK")
print(f"✅ Ollama LLM test: {test.content.strip()}")

✅ Ollama LLM test: OK


---
## Cell 4 — 50-Question Evaluation Dataset

In [4]:
EVAL_QUESTIONS = [

    # ── CONSTITUTIONAL LAW (15 questions) ────────────────────────────────
    {
        "domain": "Constitutional Law",
        "question": "What does Article 21 of the Indian Constitution guarantee?",
        "ground_truth": "Article 21 guarantees the right to life and personal liberty to every person. No person shall be deprived of their life or personal liberty except according to procedure established by law."
    },
    {
        "domain": "Constitutional Law",
        "question": "What is the significance of Article 32 of the Indian Constitution?",
        "ground_truth": "Article 32 is the right to constitutional remedies. It allows individuals to move the Supreme Court directly for enforcement of fundamental rights. Dr Ambedkar called it the heart and soul of the Constitution."
    },
    {
        "domain": "Constitutional Law",
        "question": "What are the five writs that can be issued under Article 32?",
        "ground_truth": "The Supreme Court can issue five writs under Article 32: Habeas Corpus (to produce a detained person before court), Mandamus (to compel a public authority to perform a duty), Prohibition (to prohibit a lower court from exceeding jurisdiction), Certiorari (to quash orders of inferior courts), and Quo Warranto (to challenge a person's right to hold public office)."
    },
    {
        "domain": "Constitutional Law",
        "question": "What does Article 14 of the Indian Constitution state?",
        "ground_truth": "Article 14 guarantees equality before law and equal protection of laws to all persons within the territory of India. It prohibits the state from denying any person equality before the law or the equal protection of the laws."
    },
    {
        "domain": "Constitutional Law",
        "question": "What are Directive Principles of State Policy and are they enforceable in court?",
        "ground_truth": "Directive Principles of State Policy are guidelines in Part IV (Articles 36-51) that direct the state in policy making towards a just social order. They are non-justiciable and cannot be enforced by courts, but are fundamental to governance and the duty of the state to apply them in making laws."
    },
    {
        "domain": "Constitutional Law",
        "question": "What is the difference between Fundamental Rights and Directive Principles of State Policy?",
        "ground_truth": "Fundamental Rights (Part III) are justiciable and enforceable through courts. Directive Principles (Part IV) are non-justiciable and cannot be directly enforced. Fundamental Rights are negative in nature restricting state action, while Directive Principles are positive directing state to act. Both must be read harmoniously."
    },
    {
        "domain": "Constitutional Law",
        "question": "Under what circumstances can Fundamental Rights be suspended in India?",
        "ground_truth": "Fundamental Rights can be suspended during a National Emergency declared under Article 352, except for rights under Articles 20 and 21. The President can suspend enforcement of Fundamental Rights during emergency by an order under Article 359."
    },
    {
        "domain": "Constitutional Law",
        "question": "What freedoms does Article 19 of the Indian Constitution protect?",
        "ground_truth": "Article 19 protects six freedoms: freedom of speech and expression, freedom to assemble peaceably without arms, freedom to form associations or unions, freedom to move freely throughout India, freedom to reside and settle in any part of India, and freedom to practice any profession or carry on any occupation, trade or business. These are subject to reasonable restrictions."
    },
    {
        "domain": "Constitutional Law",
        "question": "What is a Public Interest Litigation and which constitutional article enables it?",
        "ground_truth": "A PIL is a petition filed in the Supreme Court under Article 32 or High Court under Article 226 by any person in the public interest, even without a personal grievance. It was evolved by the Supreme Court in the 1980s to allow courts to enforce rights of those who cannot access justice themselves by relaxing locus standi requirements."
    },
    {
        "domain": "Constitutional Law",
        "question": "What is the procedure to amend the Indian Constitution under Article 368?",
        "ground_truth": "Article 368 provides for constitutional amendment. Most provisions require a simple majority. Some require a special majority of two-thirds of members present and voting in each House plus majority of total membership of each House. A third category also requires ratification by at least half the state legislatures, including provisions relating to federal structure, election of the President, and the Supreme Court."
    },
    {
        "domain": "Constitutional Law",
        "question": "What does the right against self-incrimination under Article 20(3) mean?",
        "ground_truth": "Article 20(3) states that no person accused of any offence shall be compelled to be a witness against himself. This protects an accused from being forced to give testimony that would incriminate them. It applies only to persons accused of offences, only to testimonial compulsion, and only in criminal proceedings."
    },
    {
        "domain": "Constitutional Law",
        "question": "What is the Basic Structure doctrine and which case established it?",
        "ground_truth": "The Basic Structure doctrine, established in Kesavananda Bharati v. State of Kerala (1973), holds that Parliament cannot amend the Constitution in a way that destroys its basic structure or essential features. The basic structure includes supremacy of the Constitution, republican and democratic form of government, secular character, separation of powers, and federal character. Amendments violating basic structure are void."
    },
    {
        "domain": "Constitutional Law",
        "question": "What rights does Article 22 provide to a person who has been arrested?",
        "ground_truth": "Article 22 provides: the right to be informed of the grounds of arrest, the right to consult and be defended by a legal practitioner of one's choice, the right to be produced before a magistrate within 24 hours of arrest, and the right not to be detained beyond 24 hours without magistrate authority. These protections do not apply to enemy aliens or persons detained under preventive detention laws."
    },
    {
        "domain": "Constitutional Law",
        "question": "What is the scope of freedom of religion under Articles 25 to 28 of the Constitution?",
        "ground_truth": "Articles 25-28 guarantee freedom of conscience and religion. Article 25 gives every person freedom to profess, practice and propagate religion subject to public order, morality and health. Article 26 gives religious denominations the right to manage their own affairs. Article 27 prohibits use of tax money for promotion of any religion. Article 28 prohibits religious instruction in state-funded institutions."
    },
    {
        "domain": "Constitutional Law",
        "question": "What are the grounds on which freedom of speech can be restricted under Article 19(2)?",
        "ground_truth": "Article 19(2) permits reasonable restrictions on freedom of speech and expression in the interests of sovereignty and integrity of India, security of the state, friendly relations with foreign states, public order, decency or morality, contempt of court, defamation, or incitement to an offence. The restrictions must be reasonable and imposed by law."
    },

    # ── IPC / CRIMINAL LAW (15 questions) ─────────────────────────────────
    {
        "domain": "Criminal Law",
        "question": "What is the definition of murder under the Indian Penal Code?",
        "ground_truth": "Under Section 300 IPC, culpable homicide is murder if the act is done with intention of causing death, intention of causing bodily injury likely to cause death, intention of causing bodily injury sufficient to cause death in ordinary course of nature, or if the person knows the act is so imminently dangerous it must cause death. Murder under Section 302 is punishable with death or imprisonment for life and fine."
    },
    {
        "domain": "Criminal Law",
        "question": "What is the difference between murder and culpable homicide not amounting to murder?",
        "ground_truth": "Culpable homicide is the genus and murder is the species. All murders are culpable homicide but not vice versa. The distinction lies in degree of intention and knowledge. Murder requires higher intent while culpable homicide not amounting to murder under Section 304 has lesser intent and carries lesser punishment of up to 10 years or life imprisonment."
    },
    {
        "domain": "Criminal Law",
        "question": "What constitutes theft under the Indian Penal Code?",
        "ground_truth": "Section 378 IPC defines theft as dishonestly taking moveable property out of the possession of any person without consent with intention to take it. Five elements: property must be moveable, in possession of another, taken without consent, with dishonest intention, and must be actually moved. Punishment under Section 379 is up to 3 years imprisonment or fine or both."
    },
    {
        "domain": "Criminal Law",
        "question": "What is the right of private defence under the IPC?",
        "ground_truth": "Sections 96-106 IPC provide the right of private defence of body and property. Section 96 states nothing is an offence done in exercise of private defence. The right is not available if there is time to seek protection of public authorities. Section 99 states it does not extend to causing more harm than necessary. Sections 100 and 103 specify when the right extends to causing death."
    },
    {
        "domain": "Criminal Law",
        "question": "What is criminal conspiracy under the IPC and how is it punished?",
        "ground_truth": "Section 120A IPC defines criminal conspiracy as an agreement between two or more persons to do an illegal act or a legal act by illegal means. Section 120B punishes it — conspiracy to commit an offence punishable with death or life imprisonment carries the same punishment as abetment of that offence. Other conspiracies carry imprisonment up to 6 months or fine or both."
    },
    {
        "domain": "Criminal Law",
        "question": "What are the essential elements of cheating under Section 420 IPC?",
        "ground_truth": "Section 420 IPC requires: fraudulent or dishonest inducement by accused, victim believing the false representation, victim delivering property or allowing it to be retained, or altering a valuable security. The offence requires deception, dishonest intent at time of inducement, and actual delivery or destruction. Punishment is up to 7 years imprisonment and fine."
    },
    {
        "domain": "Criminal Law",
        "question": "What is abetment under the IPC?",
        "ground_truth": "Section 107 IPC defines abetment as instigating a person to commit an offence, engaging in conspiracy to commit an offence, or intentionally aiding in commission of an offence. The abettor need not be present when the offence is committed. Under Section 109, if the act is committed in consequence of abetment, the abettor is punished with the same punishment as the principal offence."
    },
    {
        "domain": "Criminal Law",
        "question": "What is the difference between robbery and dacoity under the IPC?",
        "ground_truth": "Robbery under Section 390 is theft or extortion accompanied by causing or attempting to cause death, hurt, or wrongful restraint. Dacoity under Section 391 is when robbery is committed or attempted conjointly by five or more persons. The key difference is the number — robbery can be by one person while dacoity requires five or more. Dacoity carries rigorous imprisonment for 10 years to life."
    },
    {
        "domain": "Criminal Law",
        "question": "What are the eight kinds of grievous hurt listed under Section 320 IPC?",
        "ground_truth": "Section 320 IPC lists: emasculation, permanent privation of sight of either eye, permanent privation of hearing of either ear, privation of any member or joint, destruction or permanent impairing of powers of any member or joint, permanent disfiguration of head or face, fracture or dislocation of a bone or tooth, and any hurt that endangers life or causes severe bodily pain for 20 or more days."
    },
    {
        "domain": "Criminal Law",
        "question": "What are the key changes brought by the Bharatiya Nyaya Sanhita 2023 replacing the IPC?",
        "ground_truth": "BNS 2023 replaced the IPC with 358 sections versus 511. Key changes: terrorism and organised crime are codified, definition of rape is expanded, hate crimes based on religion caste and language are specifically recognised, hit-and-run is a new offence under Section 106(2), sedition under Section 124A IPC is replaced by a broader provision in Section 152 BNS, and community service is introduced as a punishment."
    },
    {
        "domain": "Criminal Law",
        "question": "What is mens rea and how is it treated under the IPC?",
        "ground_truth": "Mens rea is the mental element required for most criminal offences. Under the IPC it is expressed through words like intention, knowledge, negligence, recklessness, and dishonesty. For an act to be criminal it must be accompanied by the requisite mental state. Some offences are strict liability where mens rea is not required. Intention is the highest standard, knowledge is intermediate, and negligence is the lowest."
    },
    {
        "domain": "Criminal Law",
        "question": "What are the general exceptions to criminal liability under Chapter IV of the IPC?",
        "ground_truth": "Chapter IV Sections 76-106 provides general exceptions including: act of a child under 7 (absolute immunity), act of a child between 7-12 of immature understanding, act of an insane person, act of an intoxicated person where intoxication was involuntary, mistake of fact (not law), consent, act done by a person bound by law, act done pursuant to a court order, private defence, and act done in good faith."
    },
    {
        "domain": "Criminal Law",
        "question": "What is the punishment for attempt to commit murder under Section 307 IPC?",
        "ground_truth": "Section 307 IPC punishes attempt to commit murder with imprisonment up to 10 years and fine. If the attempt causes hurt to any person the punishment may extend to life imprisonment. If the attempt is by a person under sentence of life imprisonment the punishment may extend to death."
    },
    {
        "domain": "Criminal Law",
        "question": "What is the offence of defamation under the IPC and what are its exceptions?",
        "ground_truth": "Section 499 IPC defines defamation as making or publishing any imputation about a person intending to harm their reputation, including words signs and visible representations. Section 500 punishes it with simple imprisonment up to 2 years or fine or both. The 10 exceptions include truth published for public good, fair comment on public conduct of public servants, fair comment on literary works, report of court proceedings, and censure by a person with lawful authority."
    },
    {
        "domain": "Criminal Law",
        "question": "What is the definition of rape under the IPC after the Criminal Law Amendment Act 2013?",
        "ground_truth": "After the 2013 amendment Section 375 IPC defines rape as sexual intercourse or other specified sexual acts by a man with a woman in seven circumstances including against her will, without her consent, with consent obtained by fear or fraud, with consent when she cannot understand the nature due to intoxication or unsoundness of mind, and when she is under 18. Penetration to any extent constitutes rape. The amendment broadened the definition following the Nirbhaya case."
    },

    # ── CRIMINAL PROCEDURE / CrPC (10 questions) ──────────────────────────
    {
        "domain": "Criminal Procedure",
        "question": "What is a First Information Report and what is its legal significance?",
        "ground_truth": "An FIR is the information given to police relating to commission of a cognisable offence recorded under Section 154 CrPC. It sets criminal law in motion. It is the earliest document in a criminal case, admissible as evidence under the Evidence Act, not a substantive piece of evidence but can be used to corroborate or contradict the maker. An FIR cannot be registered for a non-cognisable offence without magistrate permission."
    },
    {
        "domain": "Criminal Procedure",
        "question": "What is anticipatory bail and how is it different from regular bail?",
        "ground_truth": "Anticipatory bail under Section 438 CrPC is bail granted in anticipation of arrest, applied for before arrest, unlike regular bail which is applied for after arrest. It is granted by Sessions Court or High Court, not Magistrate. Once granted, if the person is arrested they shall immediately be released on bail. Regular bail under Sections 436-437 is applied for after arrest from the magistrate or sessions court depending on the offence."
    },
    {
        "domain": "Criminal Procedure",
        "question": "What is the difference between a cognisable and a non-cognisable offence?",
        "ground_truth": "A cognisable offence is one where police can arrest without a warrant, investigate without magistrate permission, and register an FIR directly. Murder robbery and rape are cognisable. A non-cognisable offence is one where police cannot arrest without a warrant or investigate without a magistrate's order. Assault cheating and defamation are typically non-cognisable. The distinction determines the police's initial powers."
    },
    {
        "domain": "Criminal Procedure",
        "question": "What is a charge sheet and within what time must it be filed under CrPC?",
        "ground_truth": "A charge sheet under Section 173 CrPC is the final report submitted by police to the magistrate after completing investigation. For cognisable offences it must be filed within 60 days if the accused is in custody and the case is triable by a Sessions Court, or within 90 days in other cases. Failure to file within this period entitles the accused to default bail under Section 167(2) CrPC."
    },
    {
        "domain": "Criminal Procedure",
        "question": "What is the difference between a summons case and a warrant case under CrPC?",
        "ground_truth": "A summons case is one relating to an offence punishable with imprisonment up to 2 years. A warrant case is one relating to an offence punishable with death, life imprisonment, or imprisonment exceeding 2 years. Warrant cases have a formal framing of charges while summons cases have a simpler procedure. The distinction affects the trial procedure and available appeals."
    },
    {
        "domain": "Criminal Procedure",
        "question": "What is Section 482 CrPC and in what situations is it typically invoked?",
        "ground_truth": "Section 482 CrPC saves the inherent powers of the High Court to make orders necessary to give effect to any order under the Code, to prevent abuse of process of any court, or to secure ends of justice. It is invoked to quash FIRs where no offence is made out, to quash proceedings that are an abuse of process, to quash complaints where dispute is settled, and to prevent harassment through misuse of criminal law."
    },
    {
        "domain": "Criminal Procedure",
        "question": "What are the rights of an accused during trial under the CrPC?",
        "ground_truth": "An accused has the right to be informed of charges, right to cross-examine prosecution witnesses, right to present defence and examine own witnesses, right to be present during trial, right to legal representation, right to a copy of charge sheet and other documents, right to make a statement under Section 313 without being cross-examined, and right to appeal. The accused cannot be compelled to be a witness against themselves under Article 20(3)."
    },
    {
        "domain": "Criminal Procedure",
        "question": "Is plea bargaining available in India and for what offences?",
        "ground_truth": "Plea bargaining was introduced through Chapter XXIA Sections 265A-265L of CrPC by the 2005 amendment. It allows an accused to negotiate a reduced sentence in exchange for a guilty plea. It is available only for offences punishable with imprisonment up to 7 years, excluding offences affecting socio-economic conditions, offences against women or children, and offences punishable with death or life imprisonment. The victim must also be involved in the process."
    },
    {
        "domain": "Criminal Procedure",
        "question": "What is the procedure for filing a private complaint before a magistrate under CrPC?",
        "ground_truth": "Under Section 200 CrPC a magistrate taking cognisance on complaint shall examine the complainant and witnesses on oath. Under Section 202 the magistrate may postpone issue of process and inquire into the case or direct investigation. Under Section 203 the magistrate may dismiss the complaint if insufficient grounds exist. If sufficient grounds exist, summons or warrant is issued under Section 204."
    },
    {
        "domain": "Criminal Procedure",
        "question": "What are the key changes introduced by the Bharatiya Nagarik Suraksha Sanhita 2023?",
        "ground_truth": "BNSS 2023 replaced the CrPC with 531 sections versus 484. Key changes: trials and proceedings can be held in electronic mode, police must inform family of an arrested person, Zero FIR is codified, trials must be completed within 3 years, mercy petitions must be decided within 30 days, forensic investigation is mandatory for offences punishable with 7 or more years, and the accused must be supplied the charge sheet within 14 days of production before the magistrate."
    },

    # ── CONTRACT LAW (5 questions) ─────────────────────────────────────────
    {
        "domain": "Contract Law",
        "question": "What are the essential elements of a valid contract under the Indian Contract Act 1872?",
        "ground_truth": "Section 10 states all agreements are contracts if made by free consent of parties competent to contract, for lawful consideration and with a lawful object, and not expressly declared void. Essential elements are: offer and acceptance, free consent (not obtained by coercion, undue influence, fraud, misrepresentation or mistake), competency of parties (major, of sound mind, not disqualified), lawful consideration, lawful object, certainty of terms, and possibility of performance."
    },
    {
        "domain": "Contract Law",
        "question": "What is the difference between a void contract and a voidable contract?",
        "ground_truth": "A void contract under Section 2(j) is a contract not enforceable by law with no legal effect from the beginning. Examples include contracts with unlawful consideration, incompetent parties, or in restraint of trade. A voidable contract under Section 2(i) is valid and enforceable at the option of one party but not the other, typically where consent was obtained by coercion, undue influence, fraud or misrepresentation. The aggrieved party can choose to rescind or enforce it."
    },
    {
        "domain": "Contract Law",
        "question": "What constitutes free consent under the Indian Contract Act?",
        "ground_truth": "Section 14 defines free consent as consent not caused by coercion (Section 15 — threatening to commit an act forbidden by IPC or unlawful detaining property), undue influence (Section 16 — where one party can dominate the other's will), fraud (Section 17 — active concealment or false statement without belief in truth), misrepresentation (Section 18 — innocent false statement), or mistake (Sections 20-22 — mutual or unilateral mistake). Consent caused by any of these makes the contract voidable."
    },
    {
        "domain": "Contract Law",
        "question": "What are the remedies for breach of contract under the Indian Contract Act?",
        "ground_truth": "The main remedies are: damages under Section 73 for direct loss arising naturally from breach with the Hadley v Baxendale principle applying, suit for specific performance under the Specific Relief Act where damages are inadequate, injunction to restrain breach of negative covenant, suit for quantum meruit for reasonable compensation for work done before breach, and rescission under Section 64 relieving the party from obligations and entitling them to compensation. Section 74 covers liquidated damages clauses."
    },
    {
        "domain": "Contract Law",
        "question": "What is the difference between a contract of indemnity and a contract of guarantee?",
        "ground_truth": "A contract of indemnity under Section 124 is where one party (indemnifier) promises to save the other (indemnified) from loss caused by the indemnifier's own conduct or third party conduct. A contract of guarantee under Section 126 is a contract to perform a promise or discharge liability of a third person in case of default. In indemnity there are two parties. In guarantee there are three parties: surety, principal debtor, and creditor."
    },

    # ── EVIDENCE ACT (5 questions) ─────────────────────────────────────────
    {
        "domain": "Evidence Act",
        "question": "What is the difference between direct evidence and circumstantial evidence?",
        "ground_truth": "Direct evidence directly proves a fact in issue without any inference, such as an eyewitness account. Circumstantial evidence allows inference of the existence of a fact in issue, such as fingerprints at a scene. Both are admissible under the Indian Evidence Act. A conviction can be based solely on circumstantial evidence if the circumstances are incompatible with innocence and incapable of explanation on any other reasonable hypothesis per the Hanumant v State of MP principle."
    },
    {
        "domain": "Evidence Act",
        "question": "What is hearsay evidence and when is it admissible under the Indian Evidence Act?",
        "ground_truth": "Hearsay evidence is a statement made by someone other than the testifying witness offered to prove the truth of the matter asserted. Generally hearsay is inadmissible. Exceptions under the Indian Evidence Act include dying declarations under Section 32(1), admissions by party opponents under Section 21, statements in public documents under Section 35, entries in books of account, and statements made in the course of business under Section 32(2)."
    },
    {
        "domain": "Evidence Act",
        "question": "What is the burden of proof in criminal cases versus civil cases in India?",
        "ground_truth": "In criminal cases the burden lies on the prosecution to prove guilt beyond reasonable doubt, the highest standard. The accused is presumed innocent until proven guilty. In civil cases the burden lies on the plaintiff and the standard is preponderance of probabilities, a lower threshold. Under the Indian Evidence Act Section 101, the burden of proof means the obligation to prove a fact. The evidential burden under Section 102 may shift between parties as evidence is led."
    },
    {
        "domain": "Evidence Act",
        "question": "What is a dying declaration and what are the requirements for its admissibility?",
        "ground_truth": "A dying declaration under Section 32(1) of the Indian Evidence Act is a statement by a deceased person as to the cause of their death or circumstances of the transaction resulting in their death. Requirements: the maker must be dead, the statement must relate to cause of death or circumstances of the transaction, it is admissible regardless of whether the person expected death when making it, it must be made voluntarily without tutoring, the maker must be competent, and if incomplete or inconsistent it must be corroborated. A dying declaration alone can sustain a conviction."
    },
    {
        "domain": "Evidence Act",
        "question": "What are the rules regarding admissibility of electronic records as evidence in India?",
        "ground_truth": "Under Section 65B of the Indian Evidence Act electronic records are admissible if accompanied by a certificate from a person occupying a responsible official position in relation to the computer that produced it. The certificate must state the electronic record was produced by the computer, the computer was in regular use, the information was supplied in the ordinary course of activities, and the computer was operating properly. In Arjun Panditrao v Kailash (2020) the Supreme Court held the Section 65B certificate is mandatory."
    },
]

print(f"✅ Loaded {len(EVAL_QUESTIONS)} evaluation questions")

# Show domain distribution
from collections import Counter
dist = Counter(q["domain"] for q in EVAL_QUESTIONS)
for domain, count in dist.items():
    print(f"   {domain}: {count} questions")

✅ Loaded 50 evaluation questions
   Constitutional Law: 15 questions
   Criminal Law: 15 questions
   Criminal Procedure: 10 questions
   Contract Law: 5 questions
   Evidence Act: 5 questions


---
## Cell 5 — Query the LexRAG pipeline for each question

This calls your live FastAPI backend's `/chat` endpoint for every question and collects:
- `answer` — what LexRAG replied
- `contexts` — the chunks retrieved from Pinecone/ChromaDB

In [5]:
def query_lexrag(question: str, retries: int = 3, delay: int = 5) -> dict:
    """
    Calls the LexRAG /chat endpoint.
    Returns dict with 'answer' (str) and 'contexts' (list[str]).
    Falls back gracefully on error.
    """
    payload = {"question": question}

    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                f"{BACKEND_URL}/chat",
                json=payload,
                timeout=60,
            )
            resp.raise_for_status()
            data = resp.json()

            # ── Adjust these keys to match your actual /chat response schema ──
            # Based on your README the response has: answer, strategy, critique
            # Contexts should be in the response — add a 'contexts' field to
            # your /chat endpoint if it isn't there yet (see note below)
            answer   = data.get("answer", "")
            contexts = data.get("contexts", [])   # list of retrieved chunk strings

            # If contexts isn't returned yet, use answer as a fallback
            # (faithfulness and context_recall will be less meaningful but won't crash)
            if not contexts:
                contexts = [answer]

            return {"answer": answer, "contexts": contexts}

        except requests.exceptions.Timeout:
            print(f"   ⏱ Timeout on attempt {attempt}/{retries}")
        except requests.exceptions.HTTPError as e:
            print(f"   ❌ HTTP error on attempt {attempt}/{retries}: {e}")
        except Exception as e:
            print(f"   ❌ Error on attempt {attempt}/{retries}: {e}")

        if attempt < retries:
            time.sleep(delay)

    # All retries failed — return empty strings so eval can still run
    return {"answer": "ERROR: pipeline failed to respond", "contexts": ["ERROR"]}


# ── Run all 50 questions ──────────────────────────────────────────────────────
print("Querying LexRAG pipeline for all questions...")
print("This will take 5-15 minutes depending on your backend speed.\n")

results = []
failed  = []

for i, item in enumerate(tqdm(EVAL_QUESTIONS, desc="Querying LexRAG")):
    response = query_lexrag(item["question"])

    results.append({
        "domain":       item["domain"],
        "question":     item["question"],
        "answer":       response["answer"],
        "contexts":     response["contexts"],
        "ground_truth": item["ground_truth"],
    })

    if "ERROR" in response["answer"]:
        failed.append(i)

print(f"\n✅ Done. {len(results) - len(failed)}/{len(results)} questions answered successfully.")
if failed:
    print(f"⚠️  Failed questions (indices): {failed}")

# Save raw pipeline responses before evaluation (safety checkpoint)
raw_df = pd.DataFrame(results)
raw_df.to_csv("pipeline_responses_raw.csv", index=False)
print("💾 Raw responses saved to pipeline_responses_raw.csv")

Querying LexRAG pipeline for all questions...
This will take 5-15 minutes depending on your backend speed.



Querying LexRAG:   0%|          | 0/50 [00:00<?, ?it/s]

   ❌ HTTP error on attempt 1/3: 500 Server Error: Internal Server Error for url: http://localhost:8000/chat
   ❌ HTTP error on attempt 2/3: 500 Server Error: Internal Server Error for url: http://localhost:8000/chat


Querying LexRAG:   2%|▏         | 1/50 [00:11<09:18, 11.41s/it]

   ❌ HTTP error on attempt 3/3: 500 Server Error: Internal Server Error for url: http://localhost:8000/chat
   ❌ HTTP error on attempt 1/3: 500 Server Error: Internal Server Error for url: http://localhost:8000/chat


Querying LexRAG:   2%|▏         | 1/50 [00:16<13:44, 16.82s/it]


KeyboardInterrupt: 

---
## Cell 6 — Preview pipeline responses before evaluation

In [ ]:
# Quick preview of a few answers before running expensive evaluation
print("=" * 80)
for i in [0, 15, 30, 40, 48]:  # sample from each domain
    r = results[i]
    print(f"[{r['domain']}]")
    print(f"Q: {r['question']}")
    print(f"A: {r['answer'][:300]}..." if len(r['answer']) > 300 else f"A: {r['answer']}")
    print(f"Contexts retrieved: {len(r['contexts'])}")
    print("-" * 80)

---
## Cell 7 — Build RAGAS Dataset

In [ ]:
# Filter out any failed responses before evaluation
valid_results = [r for r in results if "ERROR" not in r["answer"]]
print(f"Building RAGAS dataset from {len(valid_results)} valid responses...")

ragas_data = {
    "question":     [r["question"]     for r in valid_results],
    "answer":       [r["answer"]       for r in valid_results],
    "contexts":     [r["contexts"]     for r in valid_results],  # list of lists
    "ground_truth": [r["ground_truth"] for r in valid_results],
}

eval_dataset = Dataset.from_dict(ragas_data)
print(f"✅ Dataset ready: {eval_dataset}")
print(f"   Columns: {eval_dataset.column_names}")
print(f"   Rows:    {len(eval_dataset)}")

---
## Cell 8 — Run RAGAS Evaluation (local Ollama judge, no rate limits)

In [ ]:
print("Starting RAGAS evaluation with local Ollama judge...")
print(f"Model: {OLLAMA_MODEL}")
print(f"Questions: {len(eval_dataset)}")
print("Estimated time: 15-45 minutes depending on your hardware.")
print("No rate limits. No API costs. Let it run.\n")

eval_start = time.time()

eval_result = evaluate(
    dataset=eval_dataset,
    metrics=[
        Faithfulness(),
        LLMContextRecall(),
        FactualCorrectness(),
        ResponseRelevancy(),
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embed,
    run_config=run_config,
    raise_exceptions=False,  # don't crash on a single bad eval — log and continue
)

elapsed = time.time() - eval_start
print(f"\n✅ Evaluation complete in {elapsed/60:.1f} minutes")
print(eval_result)

---
## Cell 9 — Save results and display summary

In [ ]:
# Convert to DataFrame with domain info attached
result_df = eval_result.to_pandas()

# Attach domain column
result_df.insert(0, "domain", [r["domain"] for r in valid_results])

# Save full per-question results
result_df.to_csv("evaluation_results.csv", index=False)
print("💾 Full results saved to evaluation_results.csv")

# ── Overall scores ─────────────────────────────────────────────────────────────
score_cols = [c for c in result_df.columns
              if c in ["faithfulness", "llm_context_recall",
                       "factual_correctness", "response_relevancy"]]

print("\n" + "="*60)
print("LEXRAG RAGAS EVALUATION RESULTS")
print(f"Judge model  : {OLLAMA_MODEL} (local Ollama)")
print(f"Questions    : {len(result_df)}")
print("="*60)

targets = {
    "faithfulness":        0.75,
    "llm_context_recall":  0.70,
    "factual_correctness": 0.65,
    "response_relevancy":  0.75,
}

overall_scores = {}
for col in score_cols:
    score  = result_df[col].mean()
    target = targets.get(col, 0.70)
    status = "✅ PASS" if score >= target else "❌ BELOW TARGET"
    overall_scores[col] = score
    print(f"{col:<30} {score:.4f}   target={target}   {status}")

print("="*60)

# ── Per-domain breakdown ───────────────────────────────────────────────────────
print("\nPER-DOMAIN BREAKDOWN")
print("-"*60)
domain_summary = result_df.groupby("domain")[score_cols].mean().round(4)
print(domain_summary.to_string())

---
## Cell 10 — Identify worst-performing questions

In [ ]:
# Find questions where faithfulness < 0.5 (hallucination risk)
if "faithfulness" in result_df.columns:
    low_faith = result_df[result_df["faithfulness"] < 0.5][["domain", "question", "faithfulness"]]
    if not low_faith.empty:
        print(f"⚠️  {len(low_faith)} questions with faithfulness < 0.5 (hallucination risk):")
        print(low_faith.to_string(index=False))
    else:
        print("✅ No questions with faithfulness < 0.5")

# Find questions where factual_correctness < 0.5
if "factual_correctness" in result_df.columns:
    low_fc = result_df[result_df["factual_correctness"] < 0.5][["domain", "question", "factual_correctness"]]
    if not low_fc.empty:
        print(f"\n⚠️  {len(low_fc)} questions with factual_correctness < 0.5:")
        print(low_fc.to_string(index=False))
    else:
        print("\n✅ No questions with factual_correctness < 0.5")

---
## Cell 11 — Generate README-ready markdown table

In [ ]:
# Copy-paste this block directly into your README.md
metric_display = {
    "faithfulness":        "Faithfulness",
    "llm_context_recall":  "Context Recall",
    "factual_correctness": "Factual Correctness",
    "response_relevancy":  "Response Relevancy",
}

lines = []
lines.append("## Evaluation Results")
lines.append("")
lines.append("Evaluated on 50 curated questions across 5 Indian legal domains.")
lines.append(f"Judge model: `{OLLAMA_MODEL}` (local Ollama — no API rate limits).")
lines.append("")
lines.append("| Metric | Score | Target | Status |")
lines.append("|---|---|---|---|")

for col, label in metric_display.items():
    if col in overall_scores:
        score  = overall_scores[col]
        target = targets.get(col, 0.70)
        status = "✅ Pass" if score >= target else "❌ Below target"
        lines.append(f"| {label} | {score:.4f} | > {target} | {status} |")

lines.append("")
lines.append("**Domain breakdown:**")
lines.append("")
lines.append("| Domain | Faithfulness | Context Recall | Factual Correctness | Response Relevancy |")
lines.append("|---|---|---|---|---|")

for domain, row in domain_summary.iterrows():
    vals = [f"{row.get(c, float('nan')):.4f}" for c in score_cols]
    lines.append(f"| {domain} | " + " | ".join(vals) + " |")

lines.append("")
lines.append("> Full results: [`evaluation_results.csv`](notebooks/evaluation_results.csv)")
lines.append("> Evaluation notebook: [`03_ragas_evaluation.ipynb`](notebooks/03_ragas_evaluation.ipynb)")

readme_block = "\n".join(lines)
print(readme_block)

# Also save it to a file
with open("evaluation_readme_block.md", "w") as f:
    f.write(readme_block)
print("\n💾 README block saved to evaluation_readme_block.md")

---
## Cell 12 — Optional: Compare Groq vs Ollama pipeline answers

Run this if you want to verify that local Ollama as eval judge gives consistent results
when compared to Groq-based answers from the pipeline.

In [ ]:
# Load previously saved results if you want to re-evaluate without
# re-running all pipeline queries

# Uncomment to reload from CSV instead of re-running Cell 5:
# raw_df = pd.read_csv("pipeline_responses_raw.csv")
# import ast
# raw_df["contexts"] = raw_df["contexts"].apply(ast.literal_eval)
# valid_results = raw_df.to_dict("records")
# print(f"Reloaded {len(valid_results)} results from CSV")

print("Tip: Once pipeline_responses_raw.csv is saved, you can re-run")
print("     RAGAS evaluation (Cells 7-11) without querying the pipeline again.")
print("     Just uncomment the lines above in this cell.")

---
## Notes for your backend

### Adding `contexts` to your `/chat` response

RAGAS needs the retrieved chunks alongside the answer to compute `faithfulness`
and `context_recall`. If your `/chat` endpoint doesn't return `contexts` yet,
add it to your `legal_graph.py` return value:

```python
# In your LangGraph Answer node — collect the retrieved chunks
return {
    "answer":   final_answer,
    "strategy": strategy,
    "critique": critique_score,
    "contexts": retrieved_chunks,  # <-- add this: list of chunk strings
}
```

```python
# In main.py /chat endpoint
@app.post("/chat")
async def chat(request: ChatRequest):
    result = graph.invoke({"question": request.question})
    return {
        "answer":   result["answer"],
        "strategy": result["strategy"],
        "critique": result["critique"],
        "contexts": result.get("contexts", []),  # <-- expose to eval
    }
```

### Install dependencies for this notebook

```bash
pip install ragas==0.2.15 langchain-ollama datasets pandas tqdm python-dotenv
```

### If Ollama is timing out

Increase the timeout in Cell 3:
```python
run_config = RunConfig(max_workers=1, timeout=180, max_retries=5)
```

Or switch to a smaller model:
```bash
ollama pull llama3.2:3b   # ~2GB, faster but less accurate eval
```
Then change `OLLAMA_MODEL = "llama3.2:3b"` in Cell 2.